In [1]:
import subprocess
import shutil

cmd = ["TotalSegmentator", "--help"]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("returncode:", result.returncode)
print("\nstdout tail:")
print(result.stdout[-2000:])

print("\nstderr tail:")
print(result.stderr[-2000:])

returncode: 0

stdout tail:
dian}, --stats_aggregation {mean,median}
                        Aggregation method for intensity statistics (default:
                        mean).
  -cp CROP_PATH, --crop_path CROP_PATH
                        Custom path to masks used for cropping. If not set
                        will use output directory.
  -bs, --body_seg       Do initial rough body segmentation and crop image to
                        body region
  -fs, --force_split    Process image in 3 chunks for less memory consumption.
                        (do not use on small images)
  -ss, --skip_saving    Skip saving of segmentations for faster runtime if you
                        are only interested in statistics.
  -ndm, --no_derived_masks
                        Do not create derived masks (e.g. skin from body
                        mask).
  -v1o, --v1_order      In multilabel file order classes as in v1. New v2
                        classes will be removed.
  -rmb, --remove_sma

In [2]:
from pathlib import Path
import SimpleITK as sitk
import numpy as np

CT_NII = Path(r"E:\jyp\ct_data_2d_preprocessed\Normal_Cases_preprocessed\ct_resampled_nii\Normal\normal001.nii.gz")

print("CT exists:", CT_NII.exists())
print("CT path:", CT_NII)

img = sitk.ReadImage(str(CT_NII))
arr = sitk.GetArrayFromImage(img)

print("size:", img.GetSize())
print("spacing:", img.GetSpacing())
print("direction:", img.GetDirection())
print("shape:", arr.shape)
print("dtype:", arr.dtype)
print("min:", arr.min())
print("max:", arr.max())
print("nan:", np.isnan(arr).sum())
print("inf:", np.isinf(arr).sum())

CT exists: True
CT path: E:\jyp\ct_data_2d_preprocessed\Normal_Cases_preprocessed\ct_resampled_nii\Normal\normal001.nii.gz
size: (512, 512, 288)
spacing: (0.71875, 0.71875, 1.0)
direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
shape: (288, 512, 512)
dtype: int16
min: -1024
max: 1697
nan: 0
inf: 0


In [ ]:
import subprocess
import shutil
import os
from pathlib import Path

CT_NII = Path(r"E:\jyp\ct_data_2d_preprocessed\Normal_Cases_preprocessed\ct_resampled_nii\Normal\normal001.nii.gz")
OUT_ROI = Path(r"E:\jyp\ts_test\normal001_lung_roi_lungprep_ts")
LOG_DIR = Path(r"E:\jyp\ts_test\jupyter_debug_lungprep_ts")

LOG_DIR.mkdir(parents=True, exist_ok=True)

if OUT_ROI.exists():
    shutil.rmtree(OUT_ROI)

cmd = [
    "TotalSegmentator",
    "-i", str(CT_NII),
    "-o", str(OUT_ROI),
    "-f",
    "--nr_thr_resamp", "1",
    "--nr_thr_saving", "1",
    "--roi_subset",
    "lung_upper_lobe_left",
    "lung_lower_lobe_left",
    "lung_upper_lobe_right",
    "lung_middle_lobe_right",
    "lung_lower_lobe_right",
]

print("========== Running TotalSegmentator ==========")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=os.environ.copy()
)

stdout_path = LOG_DIR / "lung_roi_stdout.txt"
stderr_path = LOG_DIR / "lung_roi_stderr.txt"

stdout_path.write_text(result.stdout or "", encoding="utf-8", errors="replace")
stderr_path.write_text(result.stderr or "", encoding="utf-8", errors="replace")

print("\n========== Result ==========")
print("returncode:", result.returncode)

print("\n========== stdout tail ==========")
print((result.stdout or "")[-3000:])

print("\n========== stderr tail ==========")
print((result.stderr or "")[-3000:])

print("\nstdout log:", stdout_path)
print("stderr log:", stderr_path)

print("\n========== Output check ==========")
print("OUT_ROI exists:", OUT_ROI.exists())

if OUT_ROI.exists():
    files = sorted([p.name for p in OUT_ROI.glob("*")])
    print("Output file count:", len(files))
    for f in files[:30]:
        print(f)

========== Running TotalSegmentator ==========
TotalSegmentator -i E:\jyp\ct_data_2d_preprocessed\Normal_Cases_preprocessed\ct_resampled_nii\Normal\normal001.nii.gz -o E:\jyp\ts_test\normal001_lung_roi_lungprep_ts -f --nr_thr_resamp 1 --nr_thr_saving 1 --roi_subset lung_upper_lobe_left lung_lower_lobe_left lung_upper_lobe_right lung_middle_lobe_right lung_lower_lobe_right
